### 3.3.6. Quadratic Programming and QCQP


$$\min_x\frac12x^TPx+q^Tx+r$$


**Explanation:**

Quadratic programs minimize convex quadratic objectives subject to affine constraints, while QCQPs also include quadratic constraints. Positive semidefinite curvature is the convexity requirement for the objective in a QP.

The objective contains a quadratic form $x^TPx$, a linear term $q^Tx$, and a scalar offset $r$. When the matrix $P$ is positive semidefinite, the quadratic objective is convex in the sense of [matrix definiteness](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/18_matrix_definiteness.ipynb).

**Properties:**
- A positive semidefinite Hessian gives convex curvature.
- The unconstrained optimum solves Px+q=0.
- Convex quadratic constraints require [positive semidefinite matrices](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/18_matrix_definiteness.ipynb) in the correct inequality direction.

**Intuition:**

The figure above shows quadratic contours; the calculation solves one convex quadratic stationary equation.

<img src="../../Figures/030306_bv_fig_4_5_quadratic_program_geometry.jpeg" alt="Boyd and Vandenberghe Figure 4.5: geometric illustration of a quadratic program" style="width: 60%; height: auto;">


**Numerical Example:**

The figure above shows quadratic level curves. Let
$$
P=\begin{bmatrix}2&0\\0&4\end{bmatrix},
\qquad
q=\begin{bmatrix}-2\\-8\end{bmatrix}.
$$

For
$$
f(x)=\frac12x^TPx+q^Tx,
$$
the gradient is $Px+q$. Set it to zero:
$$
Px=-q=\begin{bmatrix}2\\8\end{bmatrix}.
$$

Thus
$$
2x_1=2\Rightarrow x_1=1,
\qquad
4x_2=8\Rightarrow x_2=2.
$$

So $x^\star=(1,2)$.


In [1]:
import sympy as sp

P = sp.diag(2, 4)
q = sp.Matrix([-2, -8])
optimizer = -P.inv() * q

print("optimizer =", optimizer)


optimizer = Matrix([[1], [2]])


**Equivalent `casadi` implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
P = ca.DM([[2, 0], [0, 4]])
q = ca.DM([-2, -8])
objective = 0.5 * ca.bilin(P, decision_variable, decision_variable) + ca.dot(q, decision_variable)
gradient = ca.gradient(objective, decision_variable)
hessian = ca.jacobian(gradient, decision_variable)
optimizer = ca.solve(ca.evalf(hessian), -ca.evalf(ca.substitute(gradient, decision_variable, ca.DM.zeros(2))))

print("optimizer =", optimizer)

optimizer = [1, 2]


**Feasible-set geometry:**

A quadratic program keeps the polyhedral feasible set of a linear program but curves the objective into a convex quadratic, while a QCQP also replaces flat faces with quadratic boundaries. The first cell redraws a polyhedral feasible face, the second the convex paraboloid $z=x_1^2+x_2^2$ behind a quadratic objective, and the third an ellipsoid cut by a plane, whose flat side comes from the polyhedron and whose rounded side comes from the ellipsoid.

<img src="../../Figures/030306_qp_feasible_polyhedron_3d.png" alt="Quadratic program feasible polyhedron in three dimensions" style="width: 46%; height: auto;">
<img src="../../Figures/030306_qcqp_paraboloid.png" alt="Convex paraboloid surface for a quadratic objective" style="width: 46%; height: auto;">
<img src="../../Figures/030306_qcqp_polyhedron_ellipsoid_cap.png" alt="Intersection of a polyhedron and an ellipsoid" style="width: 46%; height: auto;">

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

simplex_vertices = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])

figure = plt.figure(figsize=(5, 5))
axis = figure.add_subplot(111, projection="3d")
axis.add_collection3d(Poly3DCollection([simplex_vertices], facecolor="#E08E2F", edgecolor="#9c5a1c", linewidth=1.2))
axis.set_xlim(0, 1)
axis.set_ylim(0, 1)
axis.set_zlim(0, 1)
axis.view_init(elev=20, azim=-120)
for pane in (axis.xaxis.pane, axis.yaxis.pane, axis.zaxis.pane):
    pane.set_facecolor((1, 1, 1, 0))
    pane.set_edgecolor("#cfcfcf")
axis.set_xticklabels([])
axis.set_yticklabels([])
axis.set_zticklabels([])
axis.tick_params(length=0)
figure.savefig("../../Figures/030306_qp_feasible_polyhedron_3d.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

bowl_colormap = LinearSegmentedColormap.from_list("bowl", ["#5a3411", "#92551b", "#cf7f29", "#efa244"])
radius = np.linspace(0, 1, 60)
angle = np.linspace(0, 2 * np.pi, 80)
radius_grid, angle_grid = np.meshgrid(radius, angle)
surface_x = radius_grid * np.cos(angle_grid)
surface_y = radius_grid * np.sin(angle_grid)
surface_z = radius_grid ** 2

figure = plt.figure(figsize=(5, 5))
axis = figure.add_subplot(111, projection="3d")
axis.plot_surface(surface_x, surface_y, surface_z, facecolors=bowl_colormap(surface_z), rcount=80, ccount=80, linewidth=0, antialiased=True, shade=False)
axis.plot(np.cos(angle), np.sin(angle), np.ones_like(angle), color="#efa244", linewidth=3)
axis.set_xlim(-1, 1)
axis.set_ylim(-1, 1)
axis.set_zlim(0, 1.05)
axis.view_init(elev=26, azim=-60)
axis.set_box_aspect((1, 1, 0.8))
for pane in (axis.xaxis.pane, axis.yaxis.pane, axis.zaxis.pane):
    pane.set_facecolor((1, 1, 1, 0))
    pane.set_edgecolor("#cfcfcf")
axis.set_xticklabels([])
axis.set_yticklabels([])
axis.set_zticklabels([])
axis.tick_params(length=0)
figure.savefig("../../Figures/030306_qcqp_paraboloid.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource

polar = np.linspace(0, np.pi, 70)
azimuth = np.linspace(-np.pi / 2, np.pi / 2, 70)
polar_grid, azimuth_grid = np.meshgrid(polar, azimuth)
semi_axis_x, semi_axis_y, semi_axis_z = 1.25, 0.8, 0.8
cap_x = semi_axis_x * np.sin(polar_grid) * np.cos(azimuth_grid)
cap_y = semi_axis_y * np.sin(polar_grid) * np.sin(azimuth_grid)
cap_z = semi_axis_z * np.cos(polar_grid)

disk_radius = np.linspace(0, 1, 30)
disk_angle = np.linspace(0, 2 * np.pi, 60)
disk_radius_grid, disk_angle_grid = np.meshgrid(disk_radius, disk_angle)
flat_x = np.zeros_like(disk_radius_grid)
flat_y = semi_axis_y * disk_radius_grid * np.cos(disk_angle_grid)
flat_z = semi_axis_z * disk_radius_grid * np.sin(disk_angle_grid)

light = LightSource(azdeg=0, altdeg=50)
figure = plt.figure(figsize=(5, 5))
axis = figure.add_subplot(111, projection="3d")
axis.plot_surface(cap_x, cap_y, cap_z, color="#E08E2F", lightsource=light, linewidth=0, shade=True)
axis.plot_surface(flat_x, flat_y, flat_z, color="#6e3f17", linewidth=0, shade=False)
axis.set_xlim(-0.1, 1.3)
axis.set_ylim(-0.9, 0.9)
axis.set_zlim(-0.85, 0.85)
axis.view_init(elev=20, azim=-110)
for pane in (axis.xaxis.pane, axis.yaxis.pane, axis.zaxis.pane):
    pane.set_facecolor((1, 1, 1, 0))
    pane.set_edgecolor("#cfcfcf")
axis.set_xticklabels([])
axis.set_yticklabels([])
axis.set_zticklabels([])
axis.tick_params(length=0)
figure.savefig("../../Figures/030306_qcqp_polyhedron_ellipsoid_cap.png", dpi=150, bbox_inches="tight")
plt.show()

**References:**

[📘 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.](https://web.stanford.edu/~boyd/cvxbook/)

---

[⬅️ Previous: Linear Programming Forms and Examples](./05_linear_programming_forms_and_examples.ipynb) | [Next: Second-Order Cone Programming ➡️](./07_second_order_cone_programming.ipynb)

---
